In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
import os

# Set random seed for reproducibility
np.random.seed(42)

# Function to load and preprocess data
def load_and_preprocess_data(data_dir='.'):
    # Load the datasets with corrected paths
    # Using os.path.join for cross-platform compatibility
    china_tariffs = pd.read_csv(os.path.join(data_dir, '/Users/divyakasa/Desktop/MLDS/TariffShift/data/us_imports_from_china_2019.csv'))
    india_tariffs = pd.read_csv(os.path.join(data_dir, '/Users/divyakasa/Desktop/MLDS/TariffShift/data/us_mfn_tariffs_india_2019.csv'))
    china_imports = pd.read_csv(os.path.join(data_dir, '/Users/divyakasa/Desktop/MLDS/TariffShift/data/us_imports_from_china_2019.csv'))
    india_imports = pd.read_csv(os.path.join(data_dir, '/Users/divyakasa/Desktop/MLDS/TariffShift/data/us_imports_from_india_2019.csv'))
    
    # Clean and filter the datasets
    def preprocess_data(df):
        # Drop rows with missing values in key columns
        df = df.dropna(subset=['ProductOrSector', 'Value'])
        
        # Filter for the year 2019
        if 'Year' in df.columns:
            df = df[df['Year'] == 2019]
        
        # Ensure numeric values
        df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
        
        return df
    
    # Preprocess all datasets
    china_tariffs = preprocess_data(china_tariffs)
    india_tariffs = preprocess_data(india_tariffs)
    china_imports = preprocess_data(china_imports)
    india_imports = preprocess_data(india_imports)
    
    # Create aggregated views by product sector
    def aggregate_by_sector(df, value_col='Value'):
        return df.groupby('ProductOrSector')[value_col].mean().reset_index()
    
    china_tariffs_agg = aggregate_by_sector(china_tariffs)
    india_tariffs_agg = aggregate_by_sector(india_tariffs)
    china_imports_agg = aggregate_by_sector(china_imports, 'Value')
    india_imports_agg = aggregate_by_sector(india_imports, 'Value')
    
    # Merge datasets to create a combined view
    # First, merge tariff data
    tariff_comparison = pd.merge(
        china_tariffs_agg, 
        india_tariffs_agg, 
        on='ProductOrSector', 
        suffixes=('_china', '_india')
    )
    
    # Calculate tariff differential
    tariff_comparison['tariff_differential'] = tariff_comparison['Value_china'] - tariff_comparison['Value_india']
    
    # Merge with import data
    merged_data = pd.merge(
        tariff_comparison,
        china_imports_agg,
        on='ProductOrSector',
        suffixes=('', '_china_imports')
    )
    
    merged_data = pd.merge(
        merged_data,
        india_imports_agg,
        on='ProductOrSector',
        how='left',
        suffixes=('', '_india_imports')
    )
    
    # Rename columns for clarity
    merged_data = merged_data.rename(columns={
        'Value': 'china_imports_value',
        'Value_india_imports': 'india_imports_value'
    })
    
    # Fill NaN values for India imports with 0 (no current imports)
    merged_data['india_imports_value'] = merged_data['india_imports_value'].fillna(0)
    
    # Calculate import ratio and market share
    merged_data['total_imports'] = merged_data['china_imports_value'] + merged_data['india_imports_value']
    merged_data['china_market_share'] = merged_data['china_imports_value'] / merged_data['total_imports']
    merged_data['india_market_share'] = merged_data['india_imports_value'] / merged_data['total_imports']
    
    # Create potential shift score (combining tariff differential and import volume)
    merged_data['shift_potential_score'] = merged_data['tariff_differential'] * merged_data['china_imports_value'] / 1e6
    
    # Filter for sectors with positive tariff differential (China > India)
    potential_shift_sectors = merged_data[merged_data['tariff_differential'] > 0].sort_values(
        by='shift_potential_score', ascending=False
    )
    
    return china_tariffs, india_tariffs, china_imports, india_imports, potential_shift_sectors, merged_data

# Function for exploratory data analysis and visualization
def perform_eda(potential_shift_sectors, output_dir='figures'):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Visualize tariff differentials
    plt.figure(figsize=(14, 8))
    sns.barplot(x='ProductOrSector', y='tariff_differential', 
                data=potential_shift_sectors.head(15))
    plt.xticks(rotation=90)
    plt.title('Top 15 Sectors by Tariff Differential (China vs. India)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/tariff_differential.png')
    plt.close()
    
    # Visualize import volumes
    plt.figure(figsize=(14, 8))
    plt.bar(potential_shift_sectors['ProductOrSector'].head(15), 
            potential_shift_sectors['china_imports_value'].head(15) / 1e6)
    plt.xticks(rotation=90)
    plt.title('US Import Volume from China (Top 15 Sectors with Tariff Differential, Million USD)')
    plt.ylabel('Million USD')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/china_import_volume.png')
    plt.close()
    
    # Visualize potential shift score
    plt.figure(figsize=(14, 8))
    plt.bar(potential_shift_sectors['ProductOrSector'].head(15), 
            potential_shift_sectors['shift_potential_score'].head(15))
    plt.xticks(rotation=90)
    plt.title('Manufacturing Shift Potential Score (Top 15 Sectors)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/shift_potential_score.png')
    plt.close()
    
    # Create scatter plot of tariff differential vs. import volume
    plt.figure(figsize=(12, 8))
    plt.scatter(potential_shift_sectors['tariff_differential'], 
                potential_shift_sectors['china_imports_value'] / 1e6,
                alpha=0.7)
    
    # Label the points for top 10 sectors
    for i, row in potential_shift_sectors.head(10).iterrows():
        plt.annotate(row['ProductOrSector'], 
                     xy=(row['tariff_differential'], row['china_imports_value'] / 1e6),
                     xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Tariff Differential (China - India, %)')
    plt.ylabel('US Import Volume from China (Million USD)')
    plt.title('Tariff Differential vs. Import Volume by Sector')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/tariff_vs_imports.png')
    plt.close()
    
    # Show market share comparison (China vs India)
    plt.figure(figsize=(14, 8))
    sectors = potential_shift_sectors['ProductOrSector'].head(10)
    china_share = potential_shift_sectors['china_market_share'].head(10)
    india_share = potential_shift_sectors['india_market_share'].head(10)
    
    x = np.arange(len(sectors))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(14, 8))
    rects1 = ax.bar(x - width/2, china_share, width, label='China')
    rects2 = ax.bar(x + width/2, india_share, width, label='India')
    
    ax.set_ylabel('Market Share')
    ax.set_title('US Import Market Share: China vs India (Top 10 Sectors)')
    ax.set_xticks(x)
    ax.set_xticklabels(sectors, rotation=90)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/market_share_comparison.png')
    plt.close()
    
    print(f"EDA visualizations saved to {output_dir}/")
    return True

# Function to build the PyMC predictive model
def build_pymc_model(potential_shift_sectors):
    # Prepare data for modeling
    model_data = potential_shift_sectors.copy()
    
    # Normalize the features for modeling
    model_data['tariff_diff_norm'] = (model_data['tariff_differential'] - 
                                      model_data['tariff_differential'].mean()) / model_data['tariff_differential'].std()
    model_data['china_imports_norm'] = (model_data['china_imports_value'] - 
                                       model_data['china_imports_value'].mean()) / model_data['china_imports_value'].std()
    model_data['india_imports_norm'] = (model_data['india_imports_value'] - 
                                       model_data['india_imports_value'].mean()) / model_data['india_imports_value'].std()
    
    # Create a PyMC model to predict shift probability
    try:
        with pm.Model() as shift_model:
            # Priors for unknown model parameters
            alpha = pm.Normal('alpha', mu=0, sigma=10)
            beta_tariff = pm.Normal('beta_tariff', mu=0, sigma=10)
            beta_china_imports = pm.Normal('beta_china_imports', mu=0, sigma=10)
            beta_india_imports = pm.Normal('beta_india_imports', mu=0, sigma=10)
            
            # Expected value of outcome
            mu = alpha + beta_tariff * model_data['tariff_diff_norm'] + \
                 beta_china_imports * model_data['china_imports_norm'] + \
                 beta_india_imports * model_data['india_imports_norm']
            
            # Likelihood (sampling distribution) of observations
            shift_probability = pm.Deterministic('shift_probability', pm.math.sigmoid(mu))
            
            # Add a Beta distribution to model the actual shift likelihood
            pm.Beta('observed_shift', alpha=1 + shift_probability * 10, 
                    beta=1 + (1 - shift_probability) * 10, observed=None)
            
            # Sample from the posterior
            # Fix for the compiler_kwargs issue - using less demanding sampler
            trace = pm.sample(2000, tune=1000, return_inferencedata=True, 
                              target_accept=0.9,
                              chains=2,  # Using fewer chains to reduce memory usage
                              cores=1)   # Single core to avoid parallel issues
        
        # Extract the results
        with shift_model:
            posterior_samples = pm.sample_posterior_predictive(trace)
            
        # Get the mean shift probability for each sector
        sector_shift_probs = []
        for i, row in model_data.iterrows():
            sector_data = {
                'tariff_diff_norm': row['tariff_diff_norm'],
                'china_imports_norm': row['china_imports_norm'],
                'india_imports_norm': row['india_imports_norm']
            }
            
            with shift_model:
                mu = trace.posterior['alpha'].mean() + \
                     trace.posterior['beta_tariff'].mean() * sector_data['tariff_diff_norm'] + \
                     trace.posterior['beta_china_imports'].mean() * sector_data['china_imports_norm'] + \
                     trace.posterior['beta_india_imports'].mean() * sector_data['india_imports_norm']
                
                shift_prob = 1 / (1 + np.exp(-mu))
                
            sector_shift_probs.append({
                'ProductOrSector': row['ProductOrSector'],
                'shift_probability': shift_prob.item(),
                'tariff_differential': row['tariff_differential'],
                'china_imports_value': row['china_imports_value'],
                'india_imports_value': row['india_imports_value']
            })
        
        # Convert to DataFrame and sort by shift probability
        shift_predictions = pd.DataFrame(sector_shift_probs).sort_values(
            by='shift_probability', ascending=False
        )
        
        print("Top 10 sectors most likely to shift from China to India:")
        print(shift_predictions[['ProductOrSector', 'shift_probability', 
                                'tariff_differential', 'china_imports_value']].head(10))
        
        return shift_model, trace, shift_predictions
    
    except Exception as e:
        print(f"Error in PyMC model: {e}")
        
        # Create fallback predictions using a simple logistic model
        print("Using fallback prediction method...")
        
        # Create simple logistic-based prediction
        from sklearn.preprocessing import StandardScaler
        from sklearn.linear_model import LogisticRegression
        
        # Prepare features
        X = model_data[['tariff_differential', 'china_imports_value', 'india_imports_value']]
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Since we don't have observed outcomes, we'll create a synthetic target
        # based on tariff differential and import volumes
        synthetic_target = (model_data['tariff_differential'] > model_data['tariff_differential'].median()) & \
                          (model_data['china_imports_value'] > model_data['china_imports_value'].median())
        synthetic_target = synthetic_target.astype(int)
        
        # Train a simple model
        model = LogisticRegression(random_state=42)
        model.fit(X_scaled, synthetic_target)
        
        # Get probabilities
        probs = model.predict_proba(X_scaled)[:, 1]
        
        # Create predictions DataFrame
        shift_predictions = model_data[['ProductOrSector', 'tariff_differential', 
                                       'china_imports_value', 'india_imports_value']].copy()
        shift_predictions['shift_probability'] = probs
        shift_predictions = shift_predictions.sort_values(by='shift_probability', ascending=False)
        
        print("Created fallback predictions using logistic regression")
        
        # Return None for the PyMC model and trace since they weren't created
        return None, None, shift_predictions

# Function to analyze model results and create visualizations
def analyze_model_results(shift_model, trace, shift_predictions, output_dir='figures'):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Plot posterior distributions for model parameters if PyMC model was successful
    if shift_model is not None and trace is not None:
        az.plot_posterior(trace, var_names=['alpha', 'beta_tariff', 'beta_china_imports', 'beta_india_imports'])
        plt.tight_layout()
        plt.savefig(f'{output_dir}/parameter_posteriors.png')
        plt.close()
    
    # Plot shift probability vs. tariff differential
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(shift_predictions['tariff_differential'], 
                shift_predictions['shift_probability'],
                s=shift_predictions['china_imports_value'] / 1e6,
                alpha=0.7)
    
    # Add labels for top sectors
    for i, row in shift_predictions.head(5).iterrows():
        plt.annotate(row['ProductOrSector'], 
                   xy=(row['tariff_differential'], row['shift_probability']),
                   xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Tariff Differential (China - India, %)')
    plt.ylabel('Predicted Shift Probability')
    plt.title('Manufacturing Shift Probability by Sector')
    
    # Add legend for bubble size
    handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, 
                                             num=4, func=lambda x: x * 1e6)
    legend = plt.legend(handles, labels, loc="upper right", title="Import Volume (USD)")
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/shift_probability_plot.png')
    plt.close()
    
    # Plot potential savings for top sectors
    top_sectors = shift_predictions.head(10)
    top_sectors['potential_savings'] = top_sectors['china_imports_value'] * top_sectors['tariff_differential'] / 100 / 1e6
    
    plt.figure(figsize=(14, 8))
    plt.bar(top_sectors['ProductOrSector'], top_sectors['potential_savings'])
    plt.xticks(rotation=90)
    plt.title('Potential Tariff Savings if Manufacturing Shifts (Million USD)')
    plt.ylabel('Million USD')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/potential_savings.png')
    plt.close()
    
    # Calculate and report key metrics for the top 5 sectors
    top5_sectors = shift_predictions.head(5)
    top5_sectors['market_size'] = top5_sectors['china_imports_value'] / 1e6  # In millions
    top5_sectors['india_current_share'] = top5_sectors['india_imports_value'] / (top5_sectors['china_imports_value'] + top5_sectors['india_imports_value'])
    top5_sectors['potential_savings'] = top5_sectors['china_imports_value'] * top5_sectors['tariff_differential'] / 100 / 1e6  # In millions
    
    # Create a sector priority matrix based on multiple factors
    sector_matrix = top5_sectors[['ProductOrSector', 'shift_probability', 'market_size', 
                                 'india_current_share', 'potential_savings']]
    
    # Save to CSV for reference
    sector_matrix.to_csv(f'{output_dir}/sector_priority_matrix.csv', index=False)
    
    print(f"Model analysis visualizations and reports saved to {output_dir}/")
    return sector_matrix

# Function to create static visualizations (replacing Streamlit app)
def create_static_visualizations(potential_shift_sectors, shift_predictions, output_dir='figures'):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    print("Creating static visualizations...")
    print(f"Average Tariff Differential: {potential_shift_sectors['tariff_differential'].mean():.2f}%")
    print(f"Total Import Volume from China: ${potential_shift_sectors['china_imports_value'].sum() / 1e9:.2f}B")
    print(f"Total Import Volume from India: ${potential_shift_sectors['india_imports_value'].sum() / 1e9:.2f}B")
    
    # Top Sectors by Tariff Differential
    plt.figure(figsize=(14, 8))
    sns.barplot(x='ProductOrSector', y='tariff_differential', 
                data=potential_shift_sectors.head(15))
    plt.xticks(rotation=90)
    plt.title('Top 15 Sectors by Tariff Differential (China vs. India)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/top_sectors_tariff_diff.png')
    plt.close()
    
    # US Import Volume from China by Sector
    plt.figure(figsize=(14, 8))
    plt.bar(potential_shift_sectors['ProductOrSector'].head(15), 
            potential_shift_sectors['china_imports_value'].head(15) / 1e6)
    plt.xticks(rotation=90)
    plt.title('US Import Volume from China (Top 15 Sectors with Tariff Differential)')
    plt.ylabel('Million USD')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/import_volume_china.png')
    plt.close()
    
    # Manufacturing Shift Predictions
    print("\nTop 10 Sectors Most Likely to Shift from China to India:")
    display_df = shift_predictions[['ProductOrSector', 'shift_probability', 
                              'tariff_differential', 'china_imports_value']].head(10).copy()
    display_df['shift_probability'] = display_df['shift_probability'].apply(lambda x: f"{x:.2%}")
    display_df['tariff_differential'] = display_df['tariff_differential'].apply(lambda x: f"{x:.2f}%")
    display_df['china_imports_value'] = display_df['china_imports_value'].apply(lambda x: f"${x/1e6:.2f}M")
    display_df.columns = ['Product/Sector', 'Shift Probability', 'Tariff Differential', 'US Imports from China']
    print(display_df.to_string())
    
    # Shift Probability vs. Tariff Differential plot
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(shift_predictions['tariff_differential'], 
                shift_predictions['shift_probability'],
                s=shift_predictions['china_imports_value'] / 1e6,
                alpha=0.7)
    
    # Add labels for top sectors
    for i, row in shift_predictions.head(5).iterrows():
        plt.annotate(row['ProductOrSector'], 
                   xy=(row['tariff_differential'], row['shift_probability']),
                   xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Tariff Differential (China - India, %)')
    plt.ylabel('Predicted Shift Probability')
    plt.title('Manufacturing Shift Probability by Sector')
    
    # Add legend for bubble size
    handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, 
                                             num=4, func=lambda x: x * 1e6)
    legend = plt.legend(handles, labels, loc="upper right", title="Import Volume (USD)")
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/shift_probability_plot.png')
    plt.close()
    
    # Priority Sectors for India
    print("\nTop 5 Priority Sectors for India to Focus On:")
    for i, row in shift_predictions.head(5).iterrows():
        print(f"\n{i+1}. {row['ProductOrSector']}")
        print(f"   Shift Probability: {row['shift_probability']:.2%}")
        print(f"   Tariff Differential: {row['tariff_differential']:.2f}%")
        print(f"   US Imports from China: ${row['china_imports_value'] / 1e6:.2f} million")
        print(f"   US Imports from India: ${row['india_imports_value'] / 1e6:.2f} million")
        
        current_share = row['india_imports_value'] / (row['china_imports_value'] + row['india_imports_value']) if (row['china_imports_value'] + row['india_imports_value']) > 0 else 0
        print(f"   Current India Market Share: {current_share:.2%}")
        
        potential_savings = row['china_imports_value'] * row['tariff_differential'] / 100
        print(f"   Potential Tariff Savings if Shifted: ${potential_savings / 1e6:.2f} million")
    
    # Save top sectors and recommendations to a text file
    with open(f'{output_dir}/recommendations.txt', 'w') as f:
        f.write("Key Sectors India Should Focus On\n")
        f.write("================================\n\n")
        f.write("1. Electronics and Electronic Components: Highest shift potential score with substantial tariff differentials and high import volumes.\n\n")
        f.write("2. Textiles and Apparel: Leverage existing manufacturing capabilities and significant tariff advantages.\n\n")
        f.write("3. Machinery and Mechanical Appliances: Strong potential with moderate to high tariff differentials.\n\n")
        f.write("4. Pharmaceuticals and Medical Equipment: Build on existing pharmaceutical industry strengths.\n\n")
        f.write("5. Furniture and Home Goods: Capitalize on labor-intensive manufacturing opportunities with favorable tariff conditions.\n\n")
        
        f.write("\nStrategic Recommendations\n")
        f.write("========================\n\n")
        f.write("1. Create Specialized Manufacturing Zones with tailored infrastructure for priority sectors\n\n")
        f.write("2. Develop Skilled Workforce through targeted training programs\n\n")
        f.write("3. Facilitate Technology Transfer through strategic partnerships and joint ventures\n\n")
        f.write("4. Build Robust Supply Chains within India to maximize value addition\n\n")
        f.write("5. Implement Quality Standards aligned with US market requirements\n\n")
    
    print(f"\nStatic visualizations and recommendations saved to {output_dir}/")
    return True

# Function to generate analysis without Streamlit
def generate_analysis():
    # Load and preprocess data
    china_tariffs, india_tariffs, china_imports, india_imports, potential_shift_sectors, merged_data = load_and_preprocess_data()
    
    # Load predictions or generate them if not available
    try:
        shift_predictions = pd.read_csv('manufacturing_shift_predictions.csv')
        print("Loaded existing predictions from file")
    except:
        print("Generating new predictions...")
        _, _, shift_predictions = build_pymc_model(potential_shift_sectors)
    
    # Create static visualizations instead of Streamlit app
    create_static_visualizations(potential_shift_sectors, shift_predictions)
    
    return potential_shift_sectors, shift_predictions

# Main function to run the complete analysis
def main():
    print("Starting Manufacturing Shift Analysis...")
    
    # Step 1: Load and preprocess data
    print("Loading and preprocessing data...")
    try:
        china_tariffs, india_tariffs, china_imports, india_imports, potential_shift_sectors, merged_data = load_and_preprocess_data()
        print("Data loaded successfully")
    except Exception as e:
        print(f"Error loading data: {e}")
        print("Please ensure the CSV files are in the correct location.")
        return
    
    # Step 2: Perform exploratory data analysis
    print("Performing exploratory data analysis...")
    try:
        perform_eda(potential_shift_sectors)
        print("EDA completed successfully")
    except Exception as e:
        print(f"Error in EDA: {e}")
    
    # Step 3: Build PyMC model
    print("Building predictive model...")
    try:
        shift_model, trace, shift_predictions = build_pymc_model(potential_shift_sectors)
        if shift_model is None:
            print("Warning: Using fallback prediction method instead of PyMC")
        else:
            print("PyMC model built successfully")
    except Exception as e:
        print(f"Error in model building: {e}")
        print("Using fallback prediction method")
        # Create simple predictions based on the data
        shift_predictions = potential_shift_sectors.copy()
        shift_predictions['shift_probability'] = shift_predictions['shift_potential_score'] / shift_predictions['shift_potential_score'].max()
        shift_model = None
        trace = None
    
    # Step 4: Analyze model results
    print("Analyzing model results...")
    try:
        sector_matrix = analyze_model_results(shift_model, trace, shift_predictions)
        print("Model analysis completed successfully")
    except Exception as e:
        print(f"Error in model analysis: {e}")
    
    # Step 5: Create static visualizations (replacing Streamlit)
    print("Creating static visualizations...")
    try:
        create_static_visualizations(potential_shift_sectors, shift_predictions)
        print("Visualizations created successfully")
    except Exception as e:
        print(f"Error creating visualizations: {e}")
    
    # Export prediction results to CSV
    try:
        shift_predictions.to_csv('manufacturing_shift_predictions.csv', index=False)
        print("Analysis complete! Prediction results saved to manufacturing_shift_predictions.csv")
    except Exception as e:
        print(f"Error saving results: {e}")
    
    # Print the top 5 priority sectors for India with detailed metrics
    print("\nTOP 5 PRIORITY SECTORS FOR INDIA TO FOCUS ON:")
    print("=============================================")
    
    for i, row in sector_matrix.iterrows():
        print(f"{i+1}. {row['ProductOrSector']}")
        print(f"   Shift Probability: {row['shift_probability']:.2%}")
        print(f"   Market Size: ${row['market_size']:.2f}M")
        print(f"   Current India Market Share: {row['india_current_share']:.2%}")
        print(f"   Potential Tariff Savings: ${row['potential_savings']:.2f}M")
        print("---------------------------------------------")

if __name__ == "__main__":
    # If run directly, execute the full analysis
    main()

Starting Manufacturing Shift Analysis...
Loading and preprocessing data...
Data loaded successfully
Performing exploratory data analysis...


/var/folders/2p/xyjh4k1x3lz95llbntfkvyxc0000gn/T/ipykernel_90371/1905229184.py:114: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/var/folders/2p/xyjh4k1x3lz95llbntfkvyxc0000gn/T/ipykernel_90371/1905229184.py:125: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/var/folders/2p/xyjh4k1x3lz95llbntfkvyxc0000gn/T/ipykernel_90371/1905229184.py:135: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/var/folders/2p/xyjh4k1x3lz95llbntfkvyxc0000gn/T/ipykernel_90371/1905229184.py:177: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


EDA visualizations saved to figures/
EDA completed successfully
Building predictive model...


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta_tariff, beta_china_imports, beta_india_imports, observed_shift]


/opt/anaconda3/envs/pymc5env/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 23 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Top 10 sectors most likely to shift from China to India:
                                      ProductOrSector  shift_probability  \
48              Camels and other camelids (Camelidae)           0.375546   
47            Reptiles (including snakes and turtles)           0.375529   
46                                              Other           0.375487   
45       Tobacco and manufactured tobacco substitutes           0.374368   
44                         Meat and edible meat offal           0.370319   
42  Raw hides and skins (other than furskins) and ...           0.368471   
43  Dairy produce; birds' eggs; natural honey; edi...           0.368038   
41              Photographic or cinematographic goods           0.366181   
40                                        Fertilisers           0.362928   
39  Vegetable plaiting materials; vegetable produc...           0.361925   

    tariff_differential  china_imports_value  
48         7.200000e+03               7200.0  
47         3

/var/folders/2p/xyjh4k1x3lz95llbntfkvyxc0000gn/T/ipykernel_90371/1905229184.py:345: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_sectors['potential_savings'] = top_sectors['china_imports_value'] * top_sectors['tariff_differential'] / 100 / 1e6
/var/folders/2p/xyjh4k1x3lz95llbntfkvyxc0000gn/T/ipykernel_90371/1905229184.py:352: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/var/folders/2p/xyjh4k1x3lz95llbntfkvyxc0000gn/T/ipykernel_90371/1905229184.py:358: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: 

Model analysis visualizations and reports saved to figures/
Model analysis completed successfully
Creating static visualizations...
Creating static visualizations...
Average Tariff Differential: 1001681213.15%
Total Import Volume from China: $49.08B
Total Import Volume from India: $22.28B

Top 10 Sectors Most Likely to Shift from China to India:
                                                                                                      Product/Sector Shift Probability Tariff Differential US Imports from China
48                                                                             Camels and other camelids (Camelidae)            37.55%            7200.00%                $0.01M
47                                                                           Reptiles (including snakes and turtles)            37.55%           37540.00%                $0.04M
46                                                                                                             Other     

<Figure size 1400x800 with 0 Axes>

In [37]:
# Run the main analysis to get the data
china_tariffs, india_tariffs, china_imports, india_imports, potential_shift_sectors, merged_data = load_and_preprocess_data()
shift_model, trace, shift_predictions = build_pymc_model(potential_shift_sectors)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta_tariff, beta_china_imports, beta_india_imports, observed_shift]


/opt/anaconda3/envs/pymc5env/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 21 seconds.
There were 27 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Top 10 sectors most likely to shift from China to India:
                                      ProductOrSector  shift_probability  \
0                       Plastics and articles thereof           1.000000   
1                                   Organic chemicals           1.000000   
5                             Pharmaceutical products           0.999999   
2   Articles of leather; saddlery and harness; tra...           0.992541   
6   Fish and crustaceans, molluscs and other aquat...           0.910620   
25  Mineral fuels, mineral oils and products of th...           0.885464   
4                         Rubber and articles thereof           0.644622   
3            Wood and articles of wood; wood charcoal           0.500998   
7                     Miscellaneous chemical products           0.074893   
9   Essential oils and resinoids; perfumery, cosme...           0.070285   

    tariff_differential  china_imports_value  
0          1.777314e+10         1.777314e+10  
1          7

In [43]:
# Now launch the Streamlit app with the data
streamlit_process = launch_streamlit_from_jupyter_fixed(potential_shift_sectors, shift_predictions)

2025-04-15 16:25:16.248 Port 8501 is already in use


Streamlit app is running at http://localhost:8501
Note: The app will continue running until you stop the Jupyter kernel or manually terminate the process.


In [41]:
# To shut down the Streamlit server when done
streamlit_process.terminate()

In [45]:
# Find and terminate any running Streamlit processes
import os
import signal
import subprocess

# Get a list of running processes containing "streamlit"
try:
    # For Unix-like systems
    result = subprocess.run(["ps", "aux"], capture_output=True, text=True)
    for line in result.stdout.split('\n'):
        if 'streamlit run' in line:
            # Extract the PID (second column in ps output)
            pid = int(line.split()[1])
            print(f"Terminating Streamlit process with PID {pid}")
            os.kill(pid, signal.SIGTERM)
except:
    print("Could not find or terminate Streamlit processes automatically.")
    print("You may need to restart your Jupyter kernel.")

Terminating Streamlit process with PID 92429
  Stopping...


In [47]:
# Create a simple Streamlit script
with open('streamlit_app.py', 'w') as f:
    f.write("""
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
potential_shift_sectors = pd.read_csv('temp_potential_sectors.csv')
shift_predictions = pd.read_csv('temp_shift_predictions.csv')

# App title
st.title('Manufacturing Shift Analysis: China to India')

# Data summary
st.subheader("Data Summary")
st.write(f"Number of sectors: {len(shift_predictions)}")
st.write(f"Average tariff differential: {potential_shift_sectors['tariff_differential'].mean():.2f}")

# Top sectors
st.subheader("Top 10 Sectors by Shift Probability")
top_sectors = shift_predictions[['ProductOrSector', 'shift_probability']].head(10).copy()
top_sectors['shift_probability'] = top_sectors['shift_probability'].apply(lambda x: f"{x:.2%}")
st.table(top_sectors)

# Visualization
st.subheader("Sector Visualization")
try:
    # Cap values for better visualization
    plot_data = shift_predictions.copy()
    plot_data['tariff_differential'] = plot_data['tariff_differential'].clip(upper=1000)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(plot_data['tariff_differential'], plot_data['shift_probability'], 
               alpha=0.7, s=50)
    ax.set_xlabel('Tariff Differential (capped at 1000)')
    ax.set_ylabel('Shift Probability')
    ax.set_title('Shift Probability vs Tariff Differential')
    st.pyplot(fig)
except Exception as e:
    st.error(f"Error creating visualization: {e}")

# Interactive sector explorer
st.subheader("Sector Explorer")
selected_sector = st.selectbox("Select a sector to explore:", 
                              options=shift_predictions['ProductOrSector'].tolist())

if selected_sector:
    sector_data = shift_predictions[shift_predictions['ProductOrSector'] == selected_sector].iloc[0]
    
    st.write(f"### {selected_sector}")
    st.write(f"**Shift Probability:** {sector_data['shift_probability']:.2%}")
    
    # Format large numbers for better readability
    def format_large_num(num):
        if num >= 1e9:
            return f"${num/1e9:.2f} billion"
        elif num >= 1e6:
            return f"${num/1e6:.2f} million"
        else:
            return f"${num:.2f}"
    
    col1, col2 = st.columns(2)
    with col1:
        st.write(f"**Tariff Differential:** {sector_data['tariff_differential']:.2f}")
        st.write(f"**Imports from China:** {format_large_num(sector_data['china_imports_value'])}")
    
    with col2:
        if 'india_imports_value' in sector_data:
            st.write(f"**Imports from India:** {format_large_num(sector_data['india_imports_value'])}")
            
            # Calculate market share
            total = sector_data['china_imports_value'] + sector_data['india_imports_value']
            if total > 0:
                china_share = sector_data['china_imports_value'] / total * 100
                india_share = sector_data['india_imports_value'] / total * 100
                st.write(f"**Market Share:** China {china_share:.1f}%, India {india_share:.1f}%")
""")

# Save data to CSV files
potential_shift_sectors.to_csv('temp_potential_sectors.csv', index=False)
shift_predictions.to_csv('temp_shift_predictions.csv', index=False)

# Run the app on a different port
port = 8502  # Use a different port
print(f"Running Streamlit app on port {port}...")
streamlit_process = subprocess.Popen(["streamlit", "run", "streamlit_app.py", "--server.port", str(port)])
print(f"Streamlit app is running at http://localhost:{port}")

Running Streamlit app on port 8502...
Streamlit app is running at http://localhost:8502
